In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-25.0.4.101-hotspot"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PostgreSQL_Test")
    .config(
        "spark.jars",
        r"C:\spark_jar\postgresql-42.7.13.jar"
    )
    .getOrCreate()
)

print("Spark:", spark.version)

c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark: 4.2.0


In [2]:
print(spark.sparkContext.getConf().get("spark.jars"))

C:\spark_jar\postgresql-42.7.13.jar


In [3]:
import os

path = "output/nyc_taxi_processed"

print("Exists:", os.path.exists(path))

if os.path.exists(path):
    print("Contents:")
    for item in os.listdir(path):
        print(item)

Exists: True
Contents:
.part-00000-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00001-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00002-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00003-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00004-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00005-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00006-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00007-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00008-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00009-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00010-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00011-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00012-8e71c0a9-e58b-49ca-9d26-736b56052717-c000.snappy.parquet.crc
.part-00013-8e71c0a9-e58b-49

In [4]:
df_final = spark.read.parquet('output/nyc_taxi_processed')

df_final.printSchema()
print(f"Rows loaded from Parquet: {df_final.count()}")

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- day_of_week: string (nu

In [6]:
df_final.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/nysc_taxi_trips_spark") \
    .option("dbtable", "consolidated_table") \
    .option("user", "postgres") \
    .option("password", "Kickboxing12#") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

print("Postgres write complete")

Postgres write complete


In [ ]:
df_pg_check = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/nysc_taxi_trips_spark") \
    .option("dbtable", "consolidated_table") \
    .option("user", "postgres") \
    .option("password", "Kickboxing12#") \
    .option("driver", "org.postgresql.Driver") \
    .load()
print(f"Postgres row count: {df_pg_check.count()}")